# Notebook 09 - Imported Component Mixed System

This lesson handles the mixed-system boundary: a custom CAD asset (STEP/STL) carries its own local coordinates, Tuba places it in global model coordinates, a programmatic pipe connects to the transformed port, and the web-scene viewer renders the relationship.

Important boundary: the mixed Code_Aster export remains a solver handoff. A mixed CAD-plus-pipe layout is not an engineering-approved result until a real mixed solve/import path is proven and the result artifacts are imported.

You will do four things:

1. Generate a local CAD asset and load it as an imported component.
2. Place the asset in global coordinates and couple a pipe to its port.
3. Inspect the local-to-global placement and coupling records.
4. Review the mixed system in the web-scene viewer.

## 1. Setup and a Local CAD Asset

Import the demo helpers and synthesize a small equipment body with a nozzle port at local origin `[0, 0, 0]`. The asset is written as STEP when gmsh is available, otherwise STL. Its coordinates live in the asset's own local CAD frame, not the Tuba model frame.

In [ ]:
from pathlib import Path
import sys
from IPython.display import HTML, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "examples").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.imported_component_mixed_system import run_demo

WORK_DIR = REPO_ROOT / "imported_component_mixed_demo"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Local CAD coordinates: the nozzle port is at local [0, 0, 0].
# The equipment body sits mostly in +X local space behind that nozzle.
LOCAL_BOUNDS = [-0.05, -0.18, -0.18, 0.45, 0.18, 0.18]
STL_PATH = WORK_DIR / "local_equipment.stl"

import trimesh

box = trimesh.creation.box(extents=(0.50, 0.36, 0.36))
box.apply_translation((0.20, 0.0, 0.0))
box.export(STL_PATH)

STEP_PATH = WORK_DIR / "local_equipment.step"
try:
    import gmsh

    gmsh.initialize()
    try:
        gmsh.model.add("local_equipment_step")
        gmsh.model.occ.addBox(*LOCAL_BOUNDS[:3], *(LOCAL_BOUNDS[i + 3] - LOCAL_BOUNDS[i] for i in range(3)))
        gmsh.model.occ.synchronize()
        gmsh.write(str(STEP_PATH))
    finally:
        gmsh.finalize()
    SOURCE_PATH = STEP_PATH
except Exception:
    SOURCE_PATH = STL_PATH

SOURCE_PATH

## 2. Place the Asset in Global Coordinates

The placement maps asset-local coordinates into the Tuba global coordinate system:

`global_point = placement.origin + placement.rotation @ local_point`

Changing `ASSET_ORIGIN_GLOBAL` moves the imported component and its confirmed port. `run_demo` builds the model, transforms the port, connects a programmatic pipe to the transformed global port position, and writes a viewer bundle.

In [ ]:
ASSET_ORIGIN_GLOBAL = [1.2, 0.45, 0.0]
ASSET_ROTATION_QWXYZ = [1.0, 0.0, 0.0, 0.0]

summary = run_demo(
    SOURCE_PATH,
    output_root=WORK_DIR / "run",
    export_study=False,
    asset_origin=ASSET_ORIGIN_GLOBAL,
    asset_rotation=ASSET_ROTATION_QWXYZ,
)
summary

## 3. Inspect the Coordinate Records

Reload the saved model and read back the placement, port, and coupling records. This confirms the local port position, its transformed global position, and which pipe node the coupling ties to.

In [ ]:
import tuba

model = tuba.Model.from_json(summary["model_path"])
asset = model.cad_assets["cad_asset_custom_equipment"]
port = model.ports["port_equipment_nozzle_a"]
coupling = model.couplings["coupling_pipe_to_equipment_a"]

{
    "asset_local_to_global_placement": asset.placement,
    "port_local_position": port.metadata["local_position"],
    "port_global_position": list(port.position),
    "pipe_endpoint_node": coupling.source_node.id,
    "coupling": coupling.to_dict(),
}

## 4. Review in the Web Scene Viewer

Launch the installed viewer directly against the generated bundle:

`tuba-viewer imported_component_mixed_demo/run/scene --open`

The launcher validates `scene.json`, chooses a local port, and opens the packaged viewer. The transparent asset body, its nozzle port, and the coupled pipe should line up at the transformed global port position.

In [ ]:
viewer_command = f'tuba-viewer "{summary["scene_dir"]}" --open'
print(viewer_command)

## Key Takeaways

- Imported CAD assets keep their own local coordinates; `placement` maps them into the Tuba global frame via `origin + rotation @ local_point`.
- A confirmed `port` transforms with the asset, so a programmatic pipe can couple to real equipment geometry.
- `cad_assets`, `ports`, and `couplings` round-trip through Tuba JSON like the rest of the model.
- The mixed Code_Aster export is a handoff only: it is not an engineering result until a real mixed solve/import path is proven and result artifacts are imported.
- The web-scene viewer renders the imported asset, its port, and the coupled pipe together for review.

Next: `10_interactive_postprocessor.ipynb` focuses on post-processing preserved Code_Aster artifacts.